# Импорт библиотек

In [1]:
# Базовые библиотеки для воспроизводимости, работы с данными и удобного вывода результатов.
import os
import sys
import random
import subprocess
from typing import List, Dict, Tuple, Optional

import numpy as np
import pandas as pd

from IPython.display import display, Markdown
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.neighbors import NearestNeighbors

os.environ["TOKENIZERS_PARALLELISM"] = "false"


def ensure_package(package_name: str, import_name: Optional[str] = None) -> None:
    """Пытается импортировать пакет и при необходимости установить его через pip."""
    target = import_name or package_name
    try:
        __import__(target)
    except Exception:
        print(f"Устанавливаем пакет: {package_name}")
        subprocess.check_call([sys.executable, "-m", "pip", "install", "-q", package_name])


# Для retrieval-контура попробуем установить основные зависимости.
# Даже если sentence-transformers не поднимется, ноутбук сможет работать через fallback.
ensure_package("faiss-cpu", "faiss")
ensure_package("sentence-transformers", "sentence_transformers")


try:
    import faiss  # type: ignore
    FAISS_AVAILABLE = True
except Exception as e:
    FAISS_AVAILABLE = False
    print("FAISS недоступен, будет использован fallback на sklearn NearestNeighbors.")
    print("Причина:", repr(e))


print("NumPy:", np.__version__)
print("Pandas:", pd.__version__)
print("FAISS available:", FAISS_AVAILABLE)

Устанавливаем пакет: faiss-cpu
Устанавливаем пакет: sentence-transformers
NumPy: 2.3.5
Pandas: 2.3.3
FAISS available: True


In [2]:
# Фиксируем seed и определяем устройство.
def set_seed(seed: int = 42) -> None:
    random.seed(seed)
    np.random.seed(seed)


set_seed(42)

try:
    import torch

    DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
except Exception:
    DEVICE = "cpu"

print("Устройство для работы:", DEVICE)

Устройство для работы: cpu


# Сборка базы данных

In [3]:
# Новая база знаний: История технологий и изобретений
documents: List[Dict[str, str]] = [
    {
        "doc_id": "doc_01",
        "title": "Изобретение транзистора",
        "text": (
            "Транзистор был изобретен в Bell Laboratories в 1947 году учеными Уильямом Шокли, Джоном Бардином и Уолтером Браттейном. "
            "Это изобретение заменило громоздкие вакуумные лампы и позволило создавать миниатюрные электронные устройства. "
            "Транзисторы стали основой для компьютеров, мобильных телефонов и всей современной электроники. "
            "За это открытие ученые получили Нобелевскую премию по физике в 1956 году."
        ),
    },
    {
        "doc_id": "doc_02",
        "title": "Создание интернета",
        "text": (
            "Интернет берет начало в сети ARPANET, созданной в 1969 году Министерством обороны США. "
            "Первоначально сеть соединяла всего 4 университетских компьютера. "
            "В 1983 году был внедрен протокол TCP/IP, который стал стандартом для передачи данных. "
            "В 1991 году Тим Бернерс-Ли изобрел World Wide Web, предложив HTTP, HTML и первые веб-браузеры, "
            "что сделало интернет доступным для широкой публики."
        ),
    },
    {
        "doc_id": "doc_03",
        "title": "Персональные компьютеры",
        "text": (
            "Первым массовым персональным компьютером стал Altair 8800 в 1975 году. "
            "В 1976 году Стив Джобс и Стив Возняк основали Apple и выпустили Apple I. "
            "В 1981 году IBM представила свой первый персональный компьютер с открытой архитектурой, "
            "что позволило другим производителям создавать совместимые устройства. "
            "Microsoft разработала MS-DOS, а затем Windows, сделав компьютеры доступными для миллионов пользователей."
        ),
    },
    {
        "doc_id": "doc_04",
        "title": "История искусственного интеллекта",
        "text": (
            "Термин 'искусственный интеллект' был предложен в 1956 году на Дартмутской конференции. "
            "В 1997 году компьютер IBM Deep Blue победил чемпиона мира по шахматам Гарри Каспарова. "
            "В 2012 году нейронные сети показали прорыв в распознавании изображений. "
            "В 2016 году программа AlphaGo от Google DeepMind победила чемпиона мира по игре Го, "
            "что считалось недостижимым для ИИ еще за несколько лет до этого."
        ),
    },
    {
        "doc_id": "doc_05",
        "title": "Развитие мобильной связи",
        "text": (
            "Первый коммерческий сотовый телефон Motorola DynaTAC 8000X появился в 1983 году. "
            "Он весил почти 1 кг и стоил около 4000 долларов. "
            "В 1992 году было отправлено первое SMS-сообщение. "
            "В 2007 году Apple представила iPhone, который совершил революцию в мобильной индустрии, "
            "предложив сенсорный экран и полноценную мобильную операционную систему."
        ),
    },
    {
        "doc_id": "doc_06",
        "title": "Изобретение микропроцессора",
        "text": (
            "Первый микропроцессор Intel 4004 был создан в 1971 году. "
            "Он содержал 2300 транзисторов и работал на частоте 740 кГц. "
            "Микропроцессоры позволили создавать программируемые калькуляторы, "
            "а затем и персональные компьютеры. "
            "Сегодня современные процессоры содержат миллиарды транзисторов и "
            "используют передовые технологии с нормами 3-5 нанометров."
        ),
    },
    {
        "doc_id": "doc_07",
        "title": "Технология Bluetooth",
        "text": (
            "Bluetooth был разработан в 1994 году компанией Ericsson как беспроводная альтернатива кабелям. "
            "Название происходит от датского короля Харальда Синезубого (Bluetooth), объединившего племена. "
            "Стандарт Bluetooth 1.0 появился в 1999 году. "
            "Сегодня Bluetooth используется в наушниках, колонках, клавиатурах и для передачи файлов. "
            "Bluetooth Low Energy (BLE) позволяет создавать устройства с очень низким энергопотреблением."
        ),
    },
    {
        "doc_id": "doc_08",
        "title": "История видеоигр",
        "text": (
            "Первой коммерческой видеоигрой стала Computer Space в 1971 году. "
            "В 1972 году Atari выпустила Pong — простейший симулятор настольного тенниса. "
            "В 1980-х Nintendo спасла индустрию после кризиса, выпустив NES и игру Super Mario Bros. "
            "В 1990-х появились 3D-игры и такие консоли, как PlayStation и Nintendo 64. "
            "Сегодня индустрия видеоигр приносит больше дохода, чем кино и музыка вместе взятые."
        ),
    },
    {
        "doc_id": "doc_09",
        "title": "Создание WWW и веб-технологий",
        "text": (
            "Тим Бернерс-Ли в 1989 году предложил концепцию Всемирной паутины, работая в CERN. "
            "Он создал первый веб-сервер, первый веб-браузер и язык HTML. "
            "В 1993 году CERN объявил, что WWW будет бесплатной для всех. "
            "Появились первые поисковые системы: WebCrawler, Yahoo, а затем Google. "
            "Сегодня веб-технологии включают HTML5, CSS3, JavaScript и множество фреймворков."
        ),
    },
    {
        "doc_id": "doc_10",
        "title": "Искусственный интеллект в повседневной жизни",
        "text": (
            "ИИ уже используется в распознавании лиц на смартфонах и в системах безопасности. "
            "Рекомендательные системы YouTube, Netflix и Spotify анализируют наши предпочтения. "
            "Голосовые помощники Siri, Alexa и Google Assistant понимают естественную речь. "
            "В медицине ИИ помогает диагностировать заболевания по снимкам МРТ и рентгена. "
            "Автопилоты Tesla используют глубокое обучение для распознавания объектов на дороге."
        ),
    },
    {
        "doc_id": "doc_11",
        "title": "Эволюция операционных систем",
        "text": (
            "В 1969 году в Bell Labs создали UNIX — многопользовательскую ОС. "
            "В 1985 году Microsoft выпустила Windows 1.0, графическую оболочку для MS-DOS. "
            "В 1991 году Линус Торвальдс создал Linux, свободную UNIX-подобную ОС. "
            "Apple в 2001 году представила Mac OS X на основе NeXTSTEP. "
            "Сегодня доминируют Windows, macOS, Linux и мобильные ОС — Android и iOS."
        ),
    },
    {
        "doc_id": "doc_12",
        "title": "Будущее технологий и тренды",
        "text": (
            "Квантовые компьютеры обещают решать задачи, недоступные классическим компьютерам. "
            "Искусственный интеллект продолжает развиваться, появляются генеративные модели, "
            "способные создавать текст, изображения и видео. "
            "Интернет вещей (IoT) соединяет миллиарды устройств. "
            "Нейроинтерфейсы, такие как разработки Neuralink, могут изменить взаимодействие "
            "человека с компьютером. "
            "Блокчейн и криптовалюты предлагают новые модели децентрализованных систем."
        ),
    },
]

docs_df = pd.DataFrame(documents)
print("Размер корпуса:", len(docs_df))
display(docs_df[["doc_id", "title"]])

Размер корпуса: 12


,doc_id,title
0,doc_01,Изобретение транзистора
1,doc_02,Создание интернета
2,doc_03,Персональные компьютеры
3,doc_04,История искусственного интеллекта
4,doc_05,Развитие мобильной связи
5,doc_06,Изобретение микропроцессора
6,doc_07,Технология Bluetooth
7,doc_08,История видеоигр
8,doc_09,Создание WWW и веб-технологий
9,doc_10,Искусственный интеллект в повседневной жизни


# Чанкинг документов

In [4]:
# Простая функция чанкинга по словам.
def chunk_text(text: str, chunk_size: int = 22, overlap: int = 5) -> List[str]:
    words = text.replace("\n", " ").split()

    if chunk_size <= 0:
        raise ValueError("chunk_size должен быть положительным.")
    if overlap >= chunk_size:
        raise ValueError("overlap должен быть меньше chunk_size.")

    chunks = []
    step = chunk_size - overlap

    for start in range(0, len(words), step):
        chunk_words = words[start : start + chunk_size]
        if not chunk_words:
            continue

        chunks.append(" ".join(chunk_words))

        if start + chunk_size >= len(words):
            break

    return chunks


def build_chunks_dataframe(
    docs: List[Dict[str, str]],
    chunk_size: int = 22,
    overlap: int = 5,
) -> pd.DataFrame:
    rows = []

    for doc in docs:
        chunks = chunk_text(doc["text"], chunk_size=chunk_size, overlap=overlap)
        for chunk_id, chunk in enumerate(chunks):
            rows.append(
                {
                    "doc_id": doc["doc_id"],
                    "title": doc["title"],
                    "chunk_id": chunk_id,
                    "chunk_text": chunk,
                    "n_words": len(chunk.split()),
                }
            )

    return pd.DataFrame(rows)


chunks_df = build_chunks_dataframe(documents, chunk_size=22, overlap=5)

print("Количество чанков:", len(chunks_df))
display(chunks_df.head(10))

Количество чанков: 38


,doc_id,title,chunk_id,chunk_text,n_words
0,doc_01,Изобретение транзистора,0,Транзистор был изобретен в Bell Laboratories в...,22
1,doc_01,Изобретение транзистора,1,Это изобретение заменило громоздкие вакуумные ...,22
2,doc_01,Изобретение транзистора,2,мобильных телефонов и всей современной электро...,18
3,doc_02,Создание интернета,0,"Интернет берет начало в сети ARPANET, созданно...",22
4,doc_02,Создание интернета,1,4 университетских компьютера. В 1983 году был ...,22
5,doc_02,Создание интернета,2,1991 году Тим Бернерс-Ли изобрел World Wide We...,21
6,doc_03,Персональные компьютеры,0,Первым массовым персональным компьютером стал ...,22
7,doc_03,Персональные компьютеры,1,Возняк основали Apple и выпустили Apple I. В 1...,22
8,doc_03,Персональные компьютеры,2,"открытой архитектурой, что позволило другим пр...",21
9,doc_04,История искусственного интеллекта,0,Термин 'искусственный интеллект' был предложен...,22


# Эмбеддинги и индекс FAISS
Выбрать модель, получить векторы, построить FAISS, показать поиск top-k.

In [5]:
# Единый интерфейс для двух вариантов векторизации: dense embeddings и fallback.
class EmbeddingBackend:
    def fit_documents(self, texts: List[str]) -> np.ndarray:
        raise NotImplementedError

    def encode_queries(self, texts: List[str]) -> np.ndarray:
        raise NotImplementedError


class SentenceTransformersBackend(EmbeddingBackend):
    def __init__(self, model_name: str, device: str = "cpu") -> None:
        from sentence_transformers import SentenceTransformer  # type: ignore

        self.model_name = model_name
        self.model = SentenceTransformer(model_name, device=device)
        self.backend_name = f"SentenceTransformer: {model_name}"

    def fit_documents(self, texts: List[str]) -> np.ndarray:
        vectors = self.model.encode(
            texts,
            batch_size=16,
            show_progress_bar=False,
            convert_to_numpy=True,
            normalize_embeddings=True,
        )
        return vectors.astype("float32")

    def encode_queries(self, texts: List[str]) -> np.ndarray:
        vectors = self.model.encode(
            texts,
            batch_size=16,
            show_progress_bar=False,
            convert_to_numpy=True,
            normalize_embeddings=True,
        )
        return vectors.astype("float32")


class TfidfFallbackBackend(EmbeddingBackend):
    def __init__(self) -> None:
        self.vectorizer = TfidfVectorizer(ngram_range=(1, 2), lowercase=True)
        self.backend_name = "TF-IDF fallback"

    def fit_documents(self, texts: List[str]) -> np.ndarray:
        vectors = self.vectorizer.fit_transform(texts).toarray()
        return vectors.astype("float32")

    def encode_queries(self, texts: List[str]) -> np.ndarray:
        vectors = self.vectorizer.transform(texts).toarray()
        return vectors.astype("float32")


def build_embedding_backend(
    model_name: str = "sentence-transformers/paraphrase-multilingual-MiniLM-L12-v2",
    device: str = "cpu",
) -> EmbeddingBackend:
    try:
        backend = SentenceTransformersBackend(model_name=model_name, device=device)
        print("Используем полноценные dense embeddings.")
        print("Бэкэнд:", backend.backend_name)
        return backend
    except Exception as e:
        print("Не удалось загрузить sentence-transformers encoder.")
        print("Причина:", repr(e))
        print("Переключаемся на TF-IDF fallback. Ноутбук останется рабочим,")
        print("но это уже не полноценные dense embeddings.")
        return TfidfFallbackBackend()


embedder = build_embedding_backend(device=DEVICE)

modules.json:   0%|          | 0.00/229 [00:00<?, ?B/s]

D:\aie_repo\rusmina-ai-system\.venv\Lib\site-packages\huggingface_hub\file_download.py:138: UserWarning: `huggingface_hub` cache-system uses symlinks by default to efficiently store duplicated files but your machine does not support them in C:\Users\EcoRestart\.cache\huggingface\hub\models--sentence-transformers--paraphrase-multilingual-MiniLM-L12-v2. Caching files will still work but in a degraded version that might require more space on your disk. This warning can be disabled by setting the `HF_HUB_DISABLE_SYMLINKS_WARNING` environment variable. For more details, see https://huggingface.co/docs/huggingface_hub/how-to-cache#limitations.
To support symlinks on Windows, you either need to activate Developer Mode or to run Python as an administrator. In order to activate developer mode, see this article: https://docs.microsoft.com/en-us/windows/apps/get-started/enable-your-device-for-development
  warnings.warn(message)


config_sentence_transformers.json:   0%|          | 0.00/122 [00:00<?, ?B/s]

README.md: 0.00B [00:00, ?B/s]

sentence_bert_config.json:   0%|          | 0.00/53.0 [00:00<?, ?B/s]

...

tokenizer_config.json:   0%|          | 0.00/526 [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/9.08M [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/239 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/190 [00:00<?, ?B/s]

Используем полноценные dense embeddings.
Бэкэнд: SentenceTransformer: sentence-transformers/paraphrase-multilingual-MiniLM-L12-v2
